In [1]:
# ==========================================================
# Import Libraries
# ==========================================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries Imported Successfully")

C:\Users\Darshan V\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Libraries Imported Successfully


In [2]:
# ==========================================================
# Load Engineered Datasets
# ==========================================================

master_df = pd.read_csv(
    "../data/final/flight_operations_cross_dataset_feature_engineered.csv"
)

weather_df = pd.read_csv(
    "../data/final/weather_cross_dataset_feature_engineered.csv"
)

delay_df = pd.read_csv(
    "../data/final/delay_cross_dataset_feature_engineered.csv"
)

print("Master Dataset :", master_df.shape)
print("Weather Dataset:", weather_df.shape)
print("Delay Dataset  :", delay_df.shape)

Master Dataset : (7516, 77)
Weather Dataset: (1629108, 21)
Delay Dataset  : (3000000, 44)


In [3]:
# ==========================================================
# Global Operational Context
# ==========================================================

weather_context = {

    "High_Weather_Risk":

    round(
        (weather_df["Weather_Operational_Severity"]=="High").mean()*100,
        2
    ),

    "Immediate_Weather_Alerts":

    round(
        (weather_df["Weather_Alert_Priority"]=="Immediate").mean()*100,
        2
    )

}

delay_context = {

    "High_Delay_Risk":

    round(
        (delay_df["Delay_Operational_Severity"]=="High").mean()*100,
        2
    ),

    "Immediate_Delay_Alerts":

    round(
        (delay_df["Delay_Alert_Priority"]=="Immediate").mean()*100,
        2
    )

}

operational_context = {

    **weather_context,

    **delay_context

}

print(operational_context)

{'High_Weather_Risk': np.float64(0.14), 'Immediate_Weather_Alerts': np.float64(0.9), 'High_Delay_Risk': np.float64(8.71), 'Immediate_Delay_Alerts': np.float64(8.71)}


In [4]:
# ==========================================================
# Flight Operational Priority Scoring
# ==========================================================

complexity_score = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

airport_score = {
    "Local Airport": 1,
    "General Airport": 2,
    "Regional Hub": 3,
    "Major Hub": 4
}

aircraft_score = {
    "Light Capability": 1,
    "Standard Capability": 2,
    "Medium Capability": 3,
    "High Capability": 4,
    "Unknown": 2
}

distance_score = {
    "Remote": 1,
    "Moderate": 2,
    "Close": 3,
    "Very Close": 4
}

In [5]:
master_df["Flight_Operational_Score"] = (

    master_df["Operational_Complexity_Level"].map(complexity_score)

    +

    master_df["Airport_Operational_Profile"].map(airport_score)

    +

    master_df["Aircraft_Operational_Capability"].map(aircraft_score)

    +

    master_df["Distance_Risk_Level"].map(distance_score)

)

master_df["Flight_Operational_Score"].describe()

count    7516.000000
mean        8.605375
std         2.258643
min         4.000000
25%         7.000000
50%         8.000000
75%        10.000000
max        15.000000
Name: Flight_Operational_Score, dtype: float64

In [6]:
def operational_priority(score):

    if score >= 13:
        return "Critical"

    elif score >= 10:
        return "High"

    elif score >= 7:
        return "Medium"

    else:
        return "Low"


master_df["Flight_Operational_Priority"] = (

    master_df["Flight_Operational_Score"]

    .apply(operational_priority)

)

master_df["Flight_Operational_Priority"].value_counts()

Flight_Operational_Priority
Medium      3836
High        1907
Low         1365
Critical     408
Name: count, dtype: int64